# 01 - Setup

Idempotent: safe to re-run. Creates

* schemas `bronze`, `silver`, `gold`, `ops` in the target catalog
* volume `ops.pipeline` for Auto Loader checkpoints and model files
* all pipeline tables (empty, `IF NOT EXISTS`)
* reference tables `ops.ref_categories`, `ops.ref_skills`, `ops.ref_india_locations` (re-seeded on every run)

**Job key** (used by every MERGE):
`sha2(concat_ws('|', lower(ats_type), lower(trim(company_name)), job_id), 256)`
where `company_name` has the `LinkedIn | ` prefix removed.

In [ ]:
import re

dbutils.widgets.text("catalog", "jobseeker", "Catalog")
CATALOG = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", CATALOG):
    raise ValueError(f"Invalid catalog name: {CATALOG!r}")

REF_VERSION = "v1"


def run(statement):
    spark.sql(statement)
    print(statement.strip().splitlines()[0])

## Schemas and volume

In [ ]:
for schema, comment in [
    ("bronze", "Raw scraper CSV rows, append only"),
    ("silver", "Cleaned, deduplicated and enriched jobs"),
    ("gold", "Curated English + India jobs for consumption"),
    ("ops", "Pipeline run logs, reference data, checkpoints"),
]:
    run(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema} COMMENT '{comment}'")

run(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.ops.pipeline COMMENT 'Auto Loader checkpoints and model files'")

for folder in ("checkpoints", "models"):
    dbutils.fs.mkdirs(f"/Volumes/{CATALOG}/ops/pipeline/{folder}")

## Bronze

In [ ]:
run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.jobs_raw (
  company_name STRING,
  ats_type STRING,
  company_url STRING,
  job_id STRING,
  title STRING,
  location STRING,
  posted_date STRING,
  job_url STRING,
  description STRING,
  collected_on STRING,
  posted_days STRING,
  _rescued_data STRING COMMENT 'Auto Loader: values that did not fit the schema',
  _source_file STRING COMMENT 'Full volume path of the CSV',
  _source_system STRING COMMENT 'Prefix of the file name: workday / greenhouse / linkedin',
  _landing_folder_date DATE COMMENT 'Date from the DDMMYYYY landing folder',
  _file_modification_time TIMESTAMP,
  _file_size BIGINT,
  _row_hash STRING COMMENT 'sha2 of all raw business columns',
  _ingested_at TIMESTAMP,
  _job_run_id STRING
)
COMMENT 'Raw scraper rows exactly as uploaded, plus ingestion metadata'
""")

## Silver

In [ ]:
run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.jobs_clean (
  job_key STRING NOT NULL COMMENT 'sha2(ats_type|company_name|job_id)',
  source STRING COMMENT 'Scraper that produced the row',
  ats_type STRING,
  company_name STRING COMMENT 'Normalized, LinkedIn prefix removed',
  company_url STRING,
  job_id STRING,
  title STRING,
  location STRING,
  country STRING,
  is_india BOOLEAN,
  posted_date DATE,
  posted_days INT,
  collected_on DATE,
  job_url STRING,
  description STRING COMMENT 'Cleaned description used for enrichment',
  description_raw_length INT,
  description_hash STRING,
  seniority_level STRING COMMENT 'LinkedIn job criteria',
  employment_type STRING COMMENT 'LinkedIn job criteria',
  job_function STRING COMMENT 'LinkedIn job criteria',
  industries STRING COMMENT 'LinkedIn job criteria',
  dq_missing_description BOOLEAN,
  first_seen_at TIMESTAMP,
  last_seen_at TIMESTAMP,
  times_seen INT,
  _source_file STRING COMMENT 'Latest file the job was seen in',
  _row_hash STRING,
  _ingested_at TIMESTAMP,
  _job_run_id STRING,
  _updated_at TIMESTAMP
)
COMMENT 'One row per job_key, latest cleaned version'
""")

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.jobs_rejected (
  reject_reason STRING,
  company_name STRING,
  ats_type STRING,
  job_id STRING,
  title STRING,
  job_url STRING,
  location STRING,
  _rescued_data STRING,
  _source_file STRING,
  _row_hash STRING,
  _ingested_at TIMESTAMP,
  _job_run_id STRING,
  _rejected_at TIMESTAMP
)
COMMENT 'Rows that failed silver validation, for debugging'
""")

# Columns added to silver.jobs_enriched after the first setup run (also added to existing tables below)
ENRICHED_EXTRA_COLUMNS = {
    "llm_status": "STRING COMMENT 'not_needed / pending / done / failed / skipped'",
    "role_title": "STRING COMMENT 'Standard role from ops.ref_roles closest to the job title + description'",
    "role_score": "DOUBLE COMMENT 'Cosine similarity of the chosen role (incl. title/category bonus)'",
    "role_alternative": "STRING COMMENT 'Second-best role'",
    "role_alternative_score": "DOUBLE",
    "role_method": "STRING COMMENT 'embedding / embedding_ambiguous (waiting for LLM tie-break) / llm'",
}
enriched_extra_ddl = ",\n  ".join(f"{name} {definition}" for name, definition in ENRICHED_EXTRA_COLUMNS.items())

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.silver.jobs_enriched (
  job_key STRING NOT NULL,
  description_hash STRING COMMENT 'Description version that was enriched',
  language STRING,
  language_confidence DOUBLE,
  experience_min_years INT,
  experience_max_years INT,
  experience_text STRING COMMENT 'Matched text the years were parsed from',
  experience_level STRING,
  skills ARRAY<STRING>,
  skill_groups ARRAY<STRING>,
  category STRING,
  category_score DOUBLE,
  category_method STRING COMMENT 'title_rule / role_embedding / role_llm',
  gold_eligible BOOLEAN,
  gold_exclusion_reason STRING,
  {enriched_extra_ddl},
  enrichment_version STRING,
  enriched_at TIMESTAMP,
  _job_run_id STRING
)
COMMENT 'Enrichment output per job_key, recomputed only when the description changes'
""")

enriched_columns = {field.name for field in spark.table(f"{CATALOG}.silver.jobs_enriched").schema.fields}
missing_enriched = [f"{name} {definition}" for name, definition in ENRICHED_EXTRA_COLUMNS.items() if name not in enriched_columns]
if missing_enriched:
    run(f"ALTER TABLE {CATALOG}.silver.jobs_enriched ADD COLUMNS ({', '.join(missing_enriched)})")

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.enrichment_llm_log (
  job_key STRING,
  description_hash STRING,
  endpoint STRING,
  prompt_version STRING COMMENT 'Responses are reused only for the same endpoint + prompt_version (p* = skills, r* = role tie-break)',
  response STRING COMMENT 'Raw JSON returned by ai_query',
  error STRING,
  called_at TIMESTAMP,
  _job_run_id STRING
)
COMMENT 'Every ai_query call made by 04_enrich; also the response cache, for debugging and quota tracking'
""")

log_columns = {field.name for field in spark.table(f"{CATALOG}.ops.enrichment_llm_log").schema.fields}
if "prompt_version" not in log_columns:
    run(f"ALTER TABLE {CATALOG}.ops.enrichment_llm_log ADD COLUMNS (prompt_version STRING COMMENT 'Responses are reused only for the same endpoint + prompt_version')")

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.embedding_cache (
  text_hash STRING NOT NULL COMMENT 'sha2 of the embedded text; identical text is never embedded twice',
  model STRING NOT NULL COMMENT 'Embedding endpoint',
  cache_key STRING COMMENT 'job_key, or role:<role_title> for reference roles',
  embedding ARRAY<FLOAT>,
  error STRING,
  created_at TIMESTAMP,
  _job_run_id STRING
)
COMMENT 'ai_query embedding cache used by 04_enrich role matching'
""")

## Gold

In [ ]:
# Tracking columns are user-owned: the gold MERGE only sets defaults on insert and never overwrites them.
# Lifecycle columns (is_active, expired_at) are set on insert by 05_gold_merge and maintained by 06_cleanup.
# Role columns come from 04_enrich; fit columns are recomputed by 05_gold_merge from ops.user_profile.
TRACKING_COLUMNS = {
    "is_applied": "BOOLEAN COMMENT 'User tracking: applied to this job'",
    "applied_at": "TIMESTAMP COMMENT 'User tracking: when applied'",
    "is_hidden": "BOOLEAN COMMENT 'User tracking: hide from lists/dashboards'",
    "hidden_at": "TIMESTAMP COMMENT 'User tracking: when hidden'",
    "application_status": "STRING COMMENT 'User tracking: not_applied / saved / applied / interviewing / offer / rejected / withdrawn'",
    "status_updated_at": "TIMESTAMP COMMENT 'User tracking: last change of any tracking column'",
    "is_active": "BOOLEAN COMMENT 'Lifecycle: false once the posting is older than the cleanup expiry window'",
    "expired_at": "TIMESTAMP COMMENT 'Lifecycle: when 06_cleanup marked the job expired'",
    "role_title": "STRING COMMENT 'Standard role (ops.ref_roles) closest to the job'",
    "role_score": "DOUBLE COMMENT 'Similarity of the chosen role'",
    "role_alternative": "STRING COMMENT 'Second-best role'",
    "role_method": "STRING COMMENT 'embedding / embedding_ambiguous / llm'",
    "fit_score": "INT COMMENT 'Fit 0-100 against ops.user_profile'",
    "fit_role": "DOUBLE COMMENT 'Fit component 0-1: role match'",
    "fit_skills": "DOUBLE COMMENT 'Fit component 0-1: skill overlap'",
    "fit_experience": "DOUBLE COMMENT 'Fit component 0-1: required years vs yours'",
    "fit_location": "DOUBLE COMMENT 'Fit component 0-1: preferred city'",
    "fit_matched_skills": "ARRAY<STRING> COMMENT 'Your skills found in the job'",
    "fit_scored_at": "TIMESTAMP COMMENT 'When the fit score last changed'",
}
tracking_ddl = ",\n  ".join(f"{name} {definition}" for name, definition in TRACKING_COLUMNS.items())

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.jobs (
  job_key STRING NOT NULL,
  source STRING,
  ats_type STRING,
  company_name STRING,
  company_url STRING,
  job_id STRING,
  title STRING,
  location STRING,
  posted_date DATE,
  posted_days INT,
  job_url STRING,
  description STRING,
  seniority_level STRING,
  employment_type STRING,
  job_function STRING,
  industries STRING,
  experience_min_years INT,
  experience_max_years INT,
  experience_level STRING,
  skills ARRAY<STRING>,
  skill_groups ARRAY<STRING>,
  category STRING,
  category_score DOUBLE,
  first_seen_at TIMESTAMP,
  last_seen_at TIMESTAMP,
  times_seen INT,
  description_hash STRING,
  enrichment_version STRING,
  {tracking_ddl},
  gold_inserted_at TIMESTAMP,
  gold_updated_at TIMESTAMP,
  _job_run_id STRING
)
COMMENT 'Curated English jobs located in India, one row per job_key (SCD Type 1) with user tracking columns'
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# Add tracking / lifecycle / role / fit columns to a gold.jobs created by an earlier version of this notebook
existing_columns = {field.name for field in spark.table(f"{CATALOG}.gold.jobs").schema.fields}
missing_columns = [f"{name} {definition}" for name, definition in TRACKING_COLUMNS.items() if name not in existing_columns]
if missing_columns:
    run(f"ALTER TABLE {CATALOG}.gold.jobs ADD COLUMNS ({', '.join(missing_columns)})")

run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.gold.job_skills (
  job_key STRING NOT NULL,
  skill STRING,
  skill_group STRING,
  category STRING,
  company_name STRING,
  posted_date DATE,
  _updated_at TIMESTAMP
)
COMMENT 'One row per job and skill, rebuilt for jobs touched in each run'
""")

## Ops

In [ ]:
run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.pipeline_runs (
  job_run_id STRING,
  task STRING,
  status STRING,
  started_at TIMESTAMP,
  finished_at TIMESTAMP,
  files_read INT,
  rows_in BIGINT,
  rows_out BIGINT,
  rows_rejected BIGINT,
  details STRING COMMENT 'JSON with task specific counters',
  message STRING
)
COMMENT 'One row per task per job run'
""")

# Your profile for the gold fit score. Seeded once; edit it with SQL, it is never overwritten by this notebook.
run(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.ops.user_profile (
  profile_id STRING NOT NULL,
  target_roles ARRAY<STRING> COMMENT 'Role titles from ops.ref_roles, most important first',
  skills ARRAY<STRING> COMMENT 'Your skills; mapped to canonical names through ops.ref_skills aliases',
  min_years INT,
  max_years INT,
  preferred_cities ARRAY<STRING> COMMENT 'Empty = any city in India scores full marks',
  weight_role DOUBLE,
  weight_skills DOUBLE,
  weight_experience DOUBLE,
  weight_location DOUBLE,
  updated_at TIMESTAMP
)
COMMENT 'Single-row profile used by 05_gold_merge to compute fit_score'
""")

if spark.table(f"{CATALOG}.ops.user_profile").count() == 0:
    run(f"""
    INSERT INTO {CATALOG}.ops.user_profile VALUES (
      'default',
      array('Data Engineer', 'Azure Data Engineer', 'Big Data Engineer', 'Databricks Engineer',
            'ETL Developer', 'Analytics Engineer', 'Data Platform Engineer'),
      array('Databricks', 'Python', 'SQL', 'PySpark', 'Cloud', 'Azure', 'Azure Data Factory'),
      2, 4,
      CAST(array() AS ARRAY<STRING>),
      0.40, 0.35, 0.15, 0.10,
      current_timestamp()
    )
    """)

## Reference data: categories
Lower `priority` is checked first by the title rules. `description` is embedded for the model-based fallback.

In [ ]:
CATEGORIES = [
    (1, "Data Engineering", "Builds data pipelines, ETL/ELT, data warehouses, data lakes and big data platforms with Spark, SQL, Kafka, Airflow, Databricks or Snowflake.",
     ["data engineer", "data engineering", "etl", "elt", "data platform", "big data", "data pipeline", "data ingestion", "data architect", "data architecture", "analytics engineer", "dataops", "data operations", "data integration", "data warehouse",
      "databricks", "snowflake", "spark", "pyspark", "informatica", "hadoop"]),
    (2, "Data Science & ML/AI", "Develops machine learning, deep learning, NLP, computer vision, generative AI and LLM models, statistical modeling and MLOps.",
     ["data scientist", "data science", "machine learning", "ml engineer", "ai engineer", "ai/ml", "ml/ai", "applied ai", "deep learning", "nlp", "computer vision", "mlops", "llm", "generative ai", "genai", "research scientist", "ai architect", "ai lead"]),
    (3, "Data Analytics & BI", "Analyzes data, builds dashboards and reports with SQL, Power BI, Tableau or Looker to produce business insights.",
     ["data analyst", "business intelligence", "bi developer", "bi engineer", "bi analyst", "reporting analyst", "analytics lead", "insights analyst", "power bi", "tableau", "product analyst"]),
    (4, "Security", "Protects systems through security engineering, SOC operations, threat detection, incident response and vulnerability management.",
     ["security", "cyber", "cybersecurity", "soc analyst", "incident responder", "detection engineer", "penetration tester", "iam engineer", "threat"]),
    (5, "QA/Testing", "Tests software quality through manual and automated testing, test frameworks and quality assurance processes.",
     ["qa", "quality assurance", "test engineer", "tester", "sdet", "test automation", "automation test"]),
    (6, "DevOps/Cloud/SRE", "Runs cloud infrastructure, CI/CD, Kubernetes, infrastructure as code, reliability engineering and platform operations.",
     ["devops", "site reliability", "sre", "cloud engineer", "platform engineer", "infrastructure engineer", "cloud architect", "kubernetes", "release engineer"]),
    (7, "Frontend/Full Stack", "Builds user facing web and mobile applications with JavaScript, React, Angular and full stack development.",
     ["frontend", "front-end", "front end", "full stack", "fullstack", "full-stack", "ui developer", "react developer", "angular developer", "web developer", "mobile developer", "android", "ios developer"]),
    (8, "Backend Engineering", "Designs and develops backend services, APIs, distributed systems and software applications in Java, Python, Go or C++.",
     ["backend", "back-end", "back end", "software engineer", "software developer", "software development engineer", "developer", "programmer", "sde", "engineering manager", "software architect", "technologist",
      "director of engineering", "head of engineering", "engineering lead"]),
    (9, "Product Management", "Owns product strategy, roadmap, requirements and prioritization as a product manager or product owner.",
     ["product manager", "product owner", "product lead", "head of product"]),
    (10, "Project/Program Management", "Plans and delivers projects and programs, manages schedules, risks and stakeholders, PMO and scrum delivery.",
     ["project manager", "program manager", "programme manager", "pmo", "scrum master", "delivery manager", "project controls", "project coordinator"]),
    (11, "Design (UX/UI)", "Creates user experience, user interface, visual and product design using Figma and design systems.",
     ["designer", "ux", "ui artist", "user experience", "visual design", "graphic design", "creative director"]),
    (12, "Finance & Accounting", "Handles accounting, financial reporting, audit, tax, treasury, FP&A, general ledger and payroll.",
     ["finance", "financial", "accountant", "accounting", "audit", "auditor", "tax", "treasury", "controller", "payroll", "fp&a", "rtr", "general ledger", "actuarial", "credit analyst"]),
    (13, "HR & People", "Manages recruiting, talent acquisition, HR operations, people partnering, learning and development.",
     ["hr", "human resources", "recruiter", "recruitment", "talent acquisition", "people operations", "people partner", "hrbp", "learning and development"]),
    (14, "Sales & Business Development", "Drives revenue through sales, account management, business development and pre-sales.",
     ["sales", "business development", "account executive", "account manager", "key account", "presales", "pre-sales", "partnerships"]),
    (15, "Marketing & Media", "Plans and runs marketing, media buying and planning, advertising, social media, SEO, content and brand campaigns.",
     ["marketing", "media", "social media", "seo", "content", "brand", "influencer", "campaign", "advertising", "programmatic", "retail search", "communications", "planning"]),
    (16, "Operations & Supply Chain", "Manages business operations, facilities, workplace services, supply chain, logistics, procurement, inventory and manufacturing.",
     ["operations", "supply chain", "logistics", "procurement", "warehouse", "inventory", "sourcing", "manufacturing",
      "facility", "facilities", "workplace", "soft services", "soft service", "front office", "housekeeping"]),
    (17, "Customer Success/Support", "Supports customers and guests through customer success, technical support, service desk and client enablement.",
     ["customer success", "customer support", "customer service", "technical support", "support engineer", "service desk", "helpdesk", "client service", "enablement",
      "guest relation", "guest relations"]),
    (18, "Consulting", "Advises clients as a consultant on business, technology or strategy engagements.",
     ["consultant", "consulting", "advisory"]),
    (99, "Other", "Any role that does not fit the other categories.", []),
]

(
    spark.createDataFrame(CATEGORIES, "priority INT, category STRING, description STRING, title_keywords ARRAY<STRING>")
    .selectExpr("*", f"'{REF_VERSION}' AS ref_version")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.ops.ref_categories")
)
print(f"ref_categories: {len(CATEGORIES)} rows")

## Reference data: standard roles
Each role belongs to one category. `04_enrich` embeds `role_title. description` with a large embedding model and assigns every gold-eligible job the closest role; the job's category is the role's category. `aliases` found in a job title give a small score bonus. Role titles must not contain single quotes.

In [ ]:
ROLES = {
    "Data Engineering": [
        ("Data Engineer", "Builds and maintains batch and streaming data pipelines, ETL/ELT jobs and data models using SQL, Python and Spark.", ["data engineer"]),
        ("Azure Data Engineer", "Data engineer on Microsoft Azure using Azure Data Factory, Synapse, Databricks, ADLS and Microsoft Fabric.", ["azure data engineer", "adf"]),
        ("AWS Data Engineer", "Data engineer on AWS using Glue, EMR, Redshift, Athena, S3, Lambda and Kinesis.", ["aws data engineer"]),
        ("GCP Data Engineer", "Data engineer on Google Cloud using BigQuery, Dataflow, Dataproc, Pub/Sub and Cloud Composer.", ["gcp data engineer", "bigquery"]),
        ("Big Data Engineer", "Processes very large datasets with Hadoop, Spark, Hive, Kafka and distributed systems.", ["big data"]),
        ("Databricks Engineer", "Builds lakehouse pipelines on Databricks with PySpark, Delta Lake, Unity Catalog and Delta Live Tables.", ["databricks"]),
        ("ETL Developer", "Develops ETL workflows with Informatica, SSIS, Talend, DataStage or ODI and SQL stored procedures.", ["etl", "informatica", "ssis", "talend"]),
        ("Analytics Engineer", "Transforms warehouse data into tested analytics models with dbt and SQL for BI consumption.", ["analytics engineer", "dbt"]),
        ("Data Platform Engineer", "Designs and operates the data platform infrastructure, orchestration, governance and tooling for data teams.", ["data platform"]),
        ("Data Architect", "Designs enterprise data architecture, data models, lakehouse and warehouse strategy and governance.", ["data architect"]),
        ("Snowflake Developer", "Builds data warehouses and pipelines on Snowflake with Snowpipe, streams, tasks and SQL.", ["snowflake"]),
        ("Streaming Data Engineer", "Builds real-time streaming pipelines with Kafka, Flink, Spark Structured Streaming and Event Hubs.", ["streaming", "kafka"]),
        ("Data Operations Engineer", "Monitors and supports production data pipelines, data quality checks and DataOps automation.", ["data operations", "dataops"]),
        ("Master Data Management Specialist", "Implements master data management, data quality and golden records with Informatica MDM, Reltio or Profisee.", ["mdm", "master data"]),
    ],
    "Data Science & ML/AI": [
        ("Data Scientist", "Builds statistical and machine learning models, experiments and insights from data with Python.", ["data scientist"]),
        ("Machine Learning Engineer", "Trains, deploys and scales machine learning models in production.", ["machine learning engineer", "ml engineer"]),
        ("AI Engineer", "Builds AI-powered applications and services integrating models and APIs.", ["ai engineer"]),
        ("Generative AI Engineer", "Builds LLM applications with RAG, prompt engineering, agents, LangChain and vector databases.", ["genai", "generative ai", "llm", "gen ai"]),
        ("MLOps Engineer", "Automates ML model training, deployment, monitoring and CI/CD with MLflow and Kubernetes.", ["mlops"]),
        ("NLP Engineer", "Builds natural language processing models for text classification, extraction and search.", ["nlp"]),
        ("Computer Vision Engineer", "Develops image and video models for detection, segmentation and recognition.", ["computer vision"]),
        ("Research Scientist", "Researches new machine learning methods and publishes or prototypes novel models.", ["research scientist"]),
        ("AI Architect", "Designs enterprise AI and ML solution architecture and platforms.", ["ai architect"]),
        ("Applied Scientist", "Applies machine learning and optimization to product problems at scale.", ["applied scientist"]),
    ],
    "Data Analytics & BI": [
        ("Data Analyst", "Analyzes data with SQL, Excel and Python to answer business questions and build reports.", ["data analyst"]),
        ("Business Intelligence Developer", "Builds BI data models, dashboards and semantic layers.", ["bi developer", "business intelligence"]),
        ("Power BI Developer", "Builds Power BI reports, DAX measures and data models.", ["power bi"]),
        ("Tableau Developer", "Builds Tableau dashboards and visual analytics.", ["tableau"]),
        ("Business Analyst", "Gathers business requirements, documents processes and bridges business and technology teams.", ["business analyst"]),
        ("Product Analyst", "Analyzes product usage, funnels and experiments to guide product decisions.", ["product analyst"]),
        ("Reporting Analyst", "Produces recurring management and operational reports and MIS.", ["reporting", "mis"]),
        ("Analytics Manager", "Leads an analytics team and owns insight delivery for the business.", ["analytics manager", "analytics lead"]),
    ],
    "Security": [
        ("Security Engineer", "Implements and operates security controls, tooling and secure architecture.", ["security engineer"]),
        ("SOC Analyst", "Monitors security alerts and investigates incidents in a security operations center using SIEM.", ["soc"]),
        ("Cloud Security Engineer", "Secures cloud platforms, identity, network and workloads on AWS, Azure or GCP.", ["cloud security"]),
        ("Security Architect", "Designs enterprise security architecture and standards.", ["security architect"]),
        ("Penetration Tester", "Performs penetration tests and vulnerability assessments.", ["penetration", "pentest"]),
        ("IAM Engineer", "Implements identity and access management, SSO and privileged access.", ["iam", "identity"]),
        ("GRC Analyst", "Manages security governance, risk, compliance and audits.", ["grc", "compliance"]),
        ("Incident Responder", "Leads investigation, containment and forensics of security incidents.", ["incident responder", "incident response"]),
        ("Detection Engineer", "Builds threat detection rules and analytics for SIEM and EDR.", ["detection engineer"]),
    ],
    "QA/Testing": [
        ("QA Engineer", "Tests software quality with test plans, manual and automated tests.", ["qa engineer", "quality assurance"]),
        ("Test Automation Engineer", "Builds automated test frameworks with Selenium, Playwright, Cypress or Appium.", ["automation", "sdet"]),
        ("Performance Test Engineer", "Runs load and performance tests with JMeter, LoadRunner or NeoLoad.", ["performance test"]),
        ("Manual Tester", "Executes manual functional and regression test cases.", ["manual test"]),
        ("QA Lead", "Leads a QA team and owns test strategy and release quality.", ["qa lead", "test lead"]),
    ],
    "DevOps/Cloud/SRE": [
        ("DevOps Engineer", "Automates build, release and infrastructure with CI/CD, Docker, Kubernetes and Terraform.", ["devops"]),
        ("Site Reliability Engineer", "Keeps production systems reliable with monitoring, SLOs, incident response and automation.", ["sre", "site reliability"]),
        ("Cloud Engineer", "Builds and operates cloud infrastructure on AWS, Azure or GCP.", ["cloud engineer"]),
        ("Cloud Architect", "Designs cloud solution architecture, landing zones and migrations.", ["cloud architect"]),
        ("Platform Engineer", "Builds internal developer platforms and shared infrastructure.", ["platform engineer"]),
        ("Kubernetes Engineer", "Operates Kubernetes clusters, containers and service meshes.", ["kubernetes"]),
        ("Infrastructure Engineer", "Manages servers, storage and infrastructure automation.", ["infrastructure"]),
        ("Network Engineer", "Designs and operates enterprise networks, routing and firewalls.", ["network engineer"]),
        ("System Administrator", "Administers Linux and Windows servers and operating systems.", ["system administrator", "sysadmin"]),
        ("Database Administrator", "Administers, tunes and backs up databases such as Oracle, SQL Server and PostgreSQL.", ["dba", "database administrator"]),
    ],
    "Frontend/Full Stack": [
        ("Frontend Developer", "Builds web user interfaces with JavaScript, TypeScript, HTML and CSS.", ["frontend", "front-end", "front end"]),
        ("React Developer", "Builds web applications with React and its ecosystem.", ["react"]),
        ("Angular Developer", "Builds web applications with Angular.", ["angular"]),
        ("Full Stack Developer", "Builds both frontend and backend of web applications.", ["full stack", "fullstack", "full-stack"]),
        ("Mobile Developer", "Builds cross-platform mobile apps with React Native or Flutter.", ["mobile", "flutter", "react native"]),
        ("Android Developer", "Builds Android apps with Kotlin or Java.", ["android"]),
        ("iOS Developer", "Builds iOS apps with Swift.", ["ios"]),
    ],
    "Backend Engineering": [
        ("Software Engineer", "Designs, builds and maintains software applications and services.", ["software engineer", "software developer", "sde"]),
        ("Backend Developer", "Builds server-side services, APIs and databases.", ["backend", "back-end", "back end"]),
        ("Java Developer", "Builds backend services with Java, Spring Boot and microservices.", ["java"]),
        ("Python Developer", "Builds applications and APIs with Python, Django, Flask or FastAPI.", ["python"]),
        (".NET Developer", "Builds applications with C#, .NET and ASP.NET.", [".net", "dotnet", "c#"]),
        ("Golang Developer", "Builds backend services in Go.", ["golang", "go developer"]),
        ("C++ Developer", "Builds high-performance or systems software in C and C++.", ["c++"]),
        ("Node.js Developer", "Builds backend services with Node.js and TypeScript.", ["node", "nodejs", "node.js"]),
        ("Salesforce Developer", "Builds Salesforce applications with Apex, LWC and integrations.", ["salesforce"]),
        ("SAP Developer", "Develops SAP solutions with ABAP, S/4HANA, Fiori and BTP.", ["sap", "abap"]),
        ("Embedded Software Engineer", "Develops firmware and embedded software for devices.", ["embedded", "firmware"]),
        ("Game Developer", "Builds games with Unreal Engine or Unity.", ["game", "unreal", "unity"]),
        ("Mainframe Developer", "Maintains mainframe applications in COBOL, JCL and DB2.", ["mainframe", "cobol"]),
        ("Engineering Manager", "Manages software engineering teams and delivery.", ["engineering manager", "director of engineering", "head of engineering"]),
        ("Software Architect", "Designs software architecture and technical direction for systems.", ["software architect", "solution architect"]),
    ],
    "Product Management": [
        ("Product Manager", "Owns product strategy, roadmap and requirements.", ["product manager"]),
        ("Technical Product Manager", "Product manager for technical, platform or API products.", ["technical product manager"]),
        ("Product Owner", "Manages the backlog and user stories for an agile team.", ["product owner"]),
        ("Head of Product", "Leads the product management organization.", ["head of product", "director of product", "vp product"]),
    ],
    "Project/Program Management": [
        ("Project Manager", "Plans and delivers projects on scope, schedule and budget.", ["project manager"]),
        ("Program Manager", "Coordinates multiple related projects and cross-team programs.", ["program manager", "programme manager"]),
        ("Scrum Master", "Facilitates agile ceremonies and removes blockers for scrum teams.", ["scrum master"]),
        ("Delivery Manager", "Owns client or product delivery and service quality.", ["delivery manager", "delivery lead"]),
        ("PMO Analyst", "Supports the project management office with governance, reporting and controls.", ["pmo", "project controls"]),
        ("Project Coordinator", "Coordinates schedules, documentation and logistics for projects.", ["project coordinator"]),
    ],
    "Design (UX/UI)": [
        ("UX Designer", "Designs user experiences, flows and wireframes.", ["ux"]),
        ("UI Designer", "Designs visual user interfaces and design systems.", ["ui designer", "ui artist"]),
        ("Product Designer", "End-to-end product design from research to high-fidelity UI in Figma.", ["product designer"]),
        ("Graphic Designer", "Creates visual and brand graphics for print and digital.", ["graphic designer"]),
        ("UX Researcher", "Plans and runs user research studies.", ["ux research", "user research"]),
        ("Technical Designer", "Implements UI and gameplay systems in game engines with scripting.", ["technical designer"]),
    ],
    "Finance & Accounting": [
        ("Accountant", "Maintains books, journal entries, reconciliations and month-end close.", ["accountant", "accounting"]),
        ("Financial Analyst", "Analyzes financial performance, models and forecasts.", ["financial analyst", "finance analyst"]),
        ("FP&A Analyst", "Owns budgeting, forecasting and planning analysis.", ["fp&a", "planning and analysis"]),
        ("Auditor", "Performs internal or external audits and control testing.", ["audit", "auditor"]),
        ("Tax Analyst", "Handles direct and indirect tax compliance and filings.", ["tax"]),
        ("Accounts Payable Specialist", "Processes vendor invoices and payments.", ["accounts payable", "ap "]),
        ("General Ledger Accountant", "Owns general ledger and record-to-report processes.", ["general ledger", "rtr", "record to report"]),
        ("Payroll Specialist", "Runs payroll processing and compliance.", ["payroll"]),
        ("Treasury Analyst", "Manages cash, liquidity and treasury operations.", ["treasury"]),
        ("Risk Analyst", "Analyzes credit, market or operational risk.", ["risk analyst", "credit risk", "market risk"]),
        ("Financial Controller", "Leads accounting operations and financial controls.", ["controller"]),
    ],
    "HR & People": [
        ("HR Generalist", "Handles broad HR operations, policies and employee relations.", ["hr generalist", "human resources"]),
        ("HR Business Partner", "Partners with business leaders on people strategy.", ["hrbp", "business partner"]),
        ("Recruiter", "Sources, screens and hires candidates.", ["recruiter", "recruitment"]),
        ("Talent Acquisition Specialist", "Runs talent acquisition programs and hiring pipelines.", ["talent acquisition"]),
        ("HR Operations Specialist", "Runs HR operations, onboarding, HRIS and employee lifecycle processes.", ["hr operations", "people operations"]),
        ("Learning and Development Specialist", "Designs and delivers training and learning programs.", ["learning and development", "l&d", "training"]),
        ("Compensation and Benefits Analyst", "Manages compensation structures and benefits programs.", ["compensation", "benefits"]),
    ],
    "Sales & Business Development": [
        ("Sales Executive", "Sells products and services and meets sales targets.", ["sales executive", "sales representative"]),
        ("Account Executive", "Closes new business deals and manages the sales cycle.", ["account executive"]),
        ("Account Manager", "Manages and grows existing customer accounts.", ["account manager", "key account"]),
        ("Business Development Manager", "Finds new business opportunities and partnerships.", ["business development"]),
        ("Pre-Sales Consultant", "Supports sales with solution demos, proposals and technical expertise.", ["presales", "pre-sales", "solution consultant"]),
        ("Inside Sales Representative", "Sells over phone and email and qualifies leads.", ["inside sales"]),
        ("Sales Manager", "Leads a sales team and territory.", ["sales manager", "head of sales"]),
    ],
    "Marketing & Media": [
        ("Digital Marketing Specialist", "Runs digital marketing campaigns across channels.", ["digital marketing"]),
        ("Performance Marketing Manager", "Manages paid search and social campaigns for ROI.", ["performance marketing", "paid"]),
        ("SEO Specialist", "Improves organic search rankings and website SEO.", ["seo"]),
        ("Content Writer", "Writes marketing and editorial content.", ["content"]),
        ("Social Media Manager", "Manages social media channels and communities.", ["social media", "social"]),
        ("Media Planner", "Plans media strategy and campaign channel mix for clients.", ["media planner", "planning", "planner"]),
        ("Programmatic Media Buyer", "Buys programmatic and digital media with DSPs such as DV360 and The Trade Desk.", ["programmatic", "media buying", "activation"]),
        ("Brand Manager", "Owns brand strategy and positioning.", ["brand"]),
        ("Marketing Analyst", "Analyzes marketing and campaign performance.", ["marketing analyst", "media analytics"]),
        ("Communications Specialist", "Manages internal or external communications and PR.", ["communications", "pr "]),
        ("Influencer Marketing Specialist", "Runs influencer and creator marketing campaigns.", ["influencer"]),
    ],
    "Operations & Supply Chain": [
        ("Operations Manager", "Manages day-to-day business operations and teams.", ["operations manager"]),
        ("Operations Analyst", "Analyzes and improves operational processes and metrics.", ["operations analyst"]),
        ("Supply Chain Analyst", "Analyzes supply chain planning, demand and inventory.", ["supply chain"]),
        ("Procurement Specialist", "Sources suppliers and manages purchasing.", ["procurement", "sourcing", "buyer"]),
        ("Logistics Coordinator", "Coordinates shipping, warehousing and transport.", ["logistics", "warehouse"]),
        ("Facility Manager", "Manages buildings, facilities and soft services.", ["facility", "facilities", "soft service"]),
        ("Workplace Coordinator", "Coordinates workplace services, front office and office administration.", ["workplace", "front office", "admin"]),
        ("Manufacturing Engineer", "Improves manufacturing processes and production lines.", ["manufacturing", "production"]),
        ("Quality Engineer", "Ensures product and process quality in manufacturing.", ["quality engineer", "quality control"]),
    ],
    "Customer Success/Support": [
        ("Customer Success Manager", "Drives customer adoption, retention and renewals.", ["customer success"]),
        ("Customer Support Specialist", "Resolves customer queries over phone, chat and email.", ["customer support", "customer service"]),
        ("Technical Support Engineer", "Troubleshoots technical product issues for customers.", ["technical support", "support engineer"]),
        ("Service Desk Analyst", "Handles IT service desk tickets and end-user support.", ["service desk", "helpdesk", "help desk"]),
        ("Client Services Executive", "Manages client relationships and service delivery.", ["client service"]),
        ("Guest Relations Executive", "Handles guest services and hospitality front desk.", ["guest relation"]),
        ("Implementation Specialist", "Onboards customers and configures the product for them.", ["implementation", "onboarding"]),
    ],
    "Consulting": [
        ("Management Consultant", "Advises clients on strategy and operations.", ["management consultant", "strategy"]),
        ("Technology Consultant", "Advises and implements technology solutions for clients.", ["technology consultant", "technical consultant"]),
        ("SAP Functional Consultant", "Configures SAP modules such as FICO, SD, MM and PP for clients.", ["sap fico", "sap sd", "sap mm", "functional consultant"]),
        ("Risk Consultant", "Advises on risk, controls, regulatory and compliance.", ["risk consultant", "risk advisory"]),
        ("Business Consultant", "Improves client business processes and operating models.", ["business consultant", "advisory"]),
    ],
}

role_rows = [
    (title, category, description, [alias.lower() for alias in aliases])
    for category, entries in ROLES.items()
    for title, description, aliases in entries
]
assert all("'" not in title for title, *_ in role_rows), "role titles must not contain single quotes"
assert len({title for title, *_ in role_rows}) == len(role_rows), "role titles must be unique"
unknown_categories = set(ROLES) - {category for _, category, _, _ in CATEGORIES}
assert not unknown_categories, f"roles reference unknown categories: {unknown_categories}"

(
    spark.createDataFrame(role_rows, "role_title STRING, category STRING, description STRING, aliases ARRAY<STRING>")
    .selectExpr("*", f"'{REF_VERSION}' AS ref_version")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.ops.ref_roles")
)
print(f"ref_roles: {len(role_rows)} roles in {len(ROLES)} categories")

## Reference data: skills
Aliases are matched case-insensitively on word boundaries. Ambiguous short names (R, Go, Unity, Lambda, Spring) only match through explicit aliases.

In [ ]:
SKILLS = {
    "Programming": [
        ("Python", ["python"]), ("Java", ["java"]), ("Scala", ["scala"]), ("SQL", ["sql", "t-sql", "tsql", "pl/sql", "plsql"]),
        ("C++", ["c++"]), ("C#", ["c#"]), (".NET", [".net", "dotnet", "asp.net"]), ("Go", ["golang"]), ("Rust", ["rust"]),
        ("JavaScript", ["javascript"]), ("TypeScript", ["typescript"]), ("Kotlin", ["kotlin"]), ("Swift", ["swift"]),
        ("PHP", ["php"]), ("Ruby", ["ruby"]), ("R", ["r programming", "rstudio", "r language"]), ("Bash", ["bash", "shell scripting"]),
        ("PowerShell", ["powershell"]), ("MATLAB", ["matlab"]), ("VBA", ["vba"]),
    ],
    "Big Data & Data Engineering": [
        ("Apache Spark", ["spark", "apache spark"]), ("PySpark", ["pyspark"]), ("Spark Streaming", ["spark streaming", "structured streaming"]),
        ("Hadoop", ["hadoop"]), ("Hive", ["hive", "hive metastore"]), ("Kafka", ["kafka", "apache kafka"]), ("Flink", ["flink", "apache flink"]),
        ("Airflow", ["airflow", "apache airflow", "cloud composer"]), ("dbt", ["dbt"]), ("Databricks", ["databricks"]), ("Delta Lake", ["delta lake"]),
        ("Unity Catalog", ["unity catalog"]), ("Snowflake", ["snowflake"]), ("Snowpipe", ["snowpipe"]), ("BigQuery", ["bigquery", "big query"]),
        ("Redshift", ["redshift"]), ("Azure Synapse", ["azure synapse", "synapse analytics", "synapse"]),
        ("Azure Data Factory", ["azure data factory", "adf"]), ("AWS Glue", ["aws glue", "glue"]), ("AWS EMR", ["aws emr", "emr"]),
        ("Athena", ["athena"]), ("Dataflow", ["dataflow"]), ("Dataproc", ["dataproc"]), ("Presto/Trino", ["presto", "trino"]),
        ("Apache Iceberg", ["iceberg", "apache iceberg"]), ("Apache Hudi", ["hudi"]), ("Parquet", ["parquet"]), ("ETL/ELT", ["etl", "elt"]),
        ("Data Warehousing", ["data warehouse", "data warehousing"]), ("Data Modeling", ["data modeling", "data modelling", "dimensional modeling", "star schema"]),
        ("Data Lake", ["data lake", "lakehouse", "adls", "adls gen2"]), ("Fivetran", ["fivetran"]), ("Informatica", ["informatica"]),
        ("Talend", ["talend"]), ("SSIS", ["ssis"]), ("SSAS", ["ssas"]), ("Matillion", ["matillion"]), ("Data Quality", ["data quality"]),
        ("Data Governance", ["data governance", "data lineage"]), ("Microsoft Purview", ["purview"]), ("Collibra", ["collibra"]),
        ("Dataiku", ["dataiku"]), ("Microsoft Fabric", ["microsoft fabric"]), ("Pub/Sub", ["pub/sub", "pubsub"]), ("Kinesis", ["kinesis"]),
        ("Event Hubs", ["event hubs", "event hub", "eventhub"]), ("Azure Data Explorer", ["azure data explorer", "kusto"]),
        ("Master Data Management", ["master data management", "mdm"]),
    ],
    "Databases": [
        ("PostgreSQL", ["postgresql", "postgres"]), ("MySQL", ["mysql"]), ("SQL Server", ["sql server", "mssql"]), ("Oracle", ["oracle"]),
        ("MongoDB", ["mongodb"]), ("Cassandra", ["cassandra"]), ("Redis", ["redis"]), ("DynamoDB", ["dynamodb"]),
        ("Elasticsearch", ["elasticsearch"]), ("OpenSearch", ["opensearch"]), ("Neo4j", ["neo4j"]), ("Cosmos DB", ["cosmos db", "cosmosdb"]),
        ("NoSQL", ["nosql"]), ("Vector Databases", ["vector database", "vector databases", "vector db", "pgvector", "faiss", "pinecone", "milvus"]),
    ],
    "Cloud": [
        ("AWS", ["aws", "amazon web services"]), ("Azure", ["azure", "microsoft azure"]), ("GCP", ["gcp", "google cloud"]),
        ("AWS S3", ["s3"]), ("AWS Lambda", ["aws lambda"]), ("AWS EC2", ["ec2"]), ("IAM", ["iam"]),
        ("Vertex AI", ["vertex ai"]), ("SageMaker", ["sagemaker"]), ("Amazon Bedrock", ["bedrock"]),
        ("Azure Machine Learning", ["azure machine learning", "azure ml"]), ("Azure OpenAI", ["azure openai"]),
    ],
    "DevOps": [
        ("Docker", ["docker"]), ("Kubernetes", ["kubernetes", "k8s"]), ("Terraform", ["terraform"]), ("CloudFormation", ["cloudformation"]),
        ("Pulumi", ["pulumi"]), ("Helm", ["helm"]), ("Jenkins", ["jenkins"]), ("GitHub Actions", ["github actions"]), ("GitLab CI", ["gitlab ci", "gitlab"]),
        ("Azure DevOps", ["azure devops"]), ("ArgoCD", ["argocd", "argo cd"]), ("CI/CD", ["ci/cd", "cicd", "continuous integration", "continuous delivery"]),
        ("Git", ["git"]), ("Linux", ["linux"]), ("Ansible", ["ansible"]), ("Prometheus", ["prometheus"]), ("Grafana", ["grafana"]),
        ("OpenTelemetry", ["opentelemetry"]), ("Datadog", ["datadog"]), ("Splunk", ["splunk"]), ("Perforce", ["perforce"]),
        ("Infrastructure as Code", ["infrastructure as code", "infrastructure-as-code", "iac"]),
    ],
    "ML & AI": [
        ("Machine Learning", ["machine learning"]), ("Deep Learning", ["deep learning"]), ("NLP", ["nlp", "natural language processing"]),
        ("Computer Vision", ["computer vision"]), ("Generative AI", ["generative ai", "genai", "gen ai"]),
        ("LLM", ["llm", "llms", "large language model", "large language models"]),
        ("RAG", ["rag", "retrieval-augmented generation", "retrieval augmented generation"]), ("Prompt Engineering", ["prompt engineering"]),
        ("AI Agents", ["ai agents", "agentic ai", "agentic", "multi-agent"]), ("LangChain", ["langchain"]), ("LangGraph", ["langgraph"]),
        ("LlamaIndex", ["llamaindex"]), ("Hugging Face", ["hugging face", "huggingface"]), ("PyTorch", ["pytorch"]),
        ("TensorFlow", ["tensorflow"]), ("Keras", ["keras"]), ("scikit-learn", ["scikit-learn", "sklearn"]), ("XGBoost", ["xgboost"]),
        ("Pandas", ["pandas"]), ("NumPy", ["numpy"]), ("MLflow", ["mlflow"]), ("MLOps", ["mlops", "llmops"]), ("Kubeflow", ["kubeflow"]),
        ("Time Series", ["time series", "forecasting"]), ("Statistics", ["statistics", "statistical modeling", "statistical analysis", "hypothesis testing"]),
        ("A/B Testing", ["a/b testing", "ab testing"]), ("Recommendation Systems", ["recommendation systems", "recommender systems"]),
        ("Reinforcement Learning", ["reinforcement learning"]), ("Graph ML", ["graph neural networks", "gnn", "graph data science"]),
        ("Model Context Protocol", ["model context protocol", "mcp"]),
    ],
    "BI & Analytics": [
        ("Power BI", ["power bi", "powerbi"]), ("Tableau", ["tableau"]), ("Looker", ["looker"]), ("Qlik", ["qlik", "qlikview", "qlik sense"]),
        ("Excel", ["excel", "ms excel", "microsoft excel"]), ("DAX", ["dax"]), ("Power Query", ["power query"]), ("SSRS", ["ssrs"]),
        ("Google Analytics", ["google analytics", "ga4"]), ("Streamlit", ["streamlit"]), ("Plotly", ["plotly"]),
    ],
    "Web & Backend": [
        ("React", ["react", "react.js", "reactjs"]), ("Angular", ["angular", "angularjs"]), ("Vue.js", ["vue", "vue.js"]),
        ("Node.js", ["node.js", "nodejs"]), ("Spring Boot", ["spring boot", "spring framework"]), ("Django", ["django"]),
        ("Flask", ["flask"]), ("FastAPI", ["fastapi"]), ("REST APIs", ["rest api", "rest apis", "restful"]), ("GraphQL", ["graphql"]),
        ("Microservices", ["microservices", "microservice"]), ("HTML/CSS", ["html", "css"]), ("Next.js", ["next.js", "nextjs"]),
    ],
    "Testing": [
        ("Selenium", ["selenium"]), ("Cypress", ["cypress"]), ("Playwright", ["playwright"]), ("JUnit", ["junit"]), ("pytest", ["pytest"]),
        ("Test Automation", ["test automation", "automation testing"]),
    ],
    "Security": [
        ("SIEM", ["siem"]), ("SOAR", ["soar"]), ("EDR/XDR", ["edr", "xdr"]), ("MITRE ATT&CK", ["mitre att&ck", "mitre"]),
        ("Threat Intelligence", ["threat intelligence"]), ("Incident Response", ["incident response"]),
        ("Penetration Testing", ["penetration testing", "pen testing"]), ("Vulnerability Management", ["vulnerability management"]),
    ],
    "Business Tools": [
        ("SAP", ["sap"]), ("Salesforce", ["salesforce"]), ("Workday", ["workday"]), ("Jira", ["jira"]), ("Confluence", ["confluence"]),
        ("ServiceNow", ["servicenow"]), ("HubSpot", ["hubspot"]), ("Salesforce Marketing Cloud", ["marketing cloud"]),
        ("Google Ads", ["google ads"]), ("Meta Ads", ["meta ads", "facebook ads"]), ("DV360", ["dv360", "dv 360"]),
        ("The Trade Desk", ["the trade desk"]), ("Amazon Ads", ["amazon ads"]), ("Figma", ["figma"]), ("Photoshop", ["photoshop"]),
        ("Illustrator", ["illustrator"]), ("Unreal Engine", ["unreal engine", "ue5", "ue4"]), ("Unity (game engine)", ["unity3d", "unity engine"]),
        ("Primavera P6", ["primavera", "p6"]), ("Agile", ["agile"]), ("Scrum", ["scrum"]),
    ],
}

skill_rows = [(skill, group, aliases) for group, entries in SKILLS.items() for skill, aliases in entries]
(
    spark.createDataFrame(skill_rows, "skill STRING, skill_group STRING, aliases ARRAY<STRING>")
    .selectExpr("*", f"'{REF_VERSION}' AS ref_version")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.ops.ref_skills")
)
print(f"ref_skills: {len(skill_rows)} rows")

## Reference data: India locations
Used to detect India when the location has no country (e.g. `Gurgaon`).

In [ ]:
INDIA_LOCATIONS = [
    ("india", "India", None),
    ("bengaluru", "Bengaluru", "Karnataka"), ("bangalore", "Bengaluru", "Karnataka"), ("mysuru", "Mysuru", "Karnataka"), ("mysore", "Mysuru", "Karnataka"),
    ("mangaluru", "Mangaluru", "Karnataka"), ("mangalore", "Mangaluru", "Karnataka"),
    ("hyderabad", "Hyderabad", "Telangana"), ("secunderabad", "Hyderabad", "Telangana"),
    ("chennai", "Chennai", "Tamil Nadu"), ("madras", "Chennai", "Tamil Nadu"), ("coimbatore", "Coimbatore", "Tamil Nadu"), ("madurai", "Madurai", "Tamil Nadu"),
    ("mumbai", "Mumbai", "Maharashtra"), ("bombay", "Mumbai", "Maharashtra"), ("navi mumbai", "Navi Mumbai", "Maharashtra"), ("thane", "Thane", "Maharashtra"),
    ("pune", "Pune", "Maharashtra"), ("pimpri-chinchwad", "Pune", "Maharashtra"), ("nagpur", "Nagpur", "Maharashtra"), ("nashik", "Nashik", "Maharashtra"),
    ("aurangabad", "Aurangabad", "Maharashtra"),
    ("delhi", "Delhi", "Delhi"), ("new delhi", "Delhi", "Delhi"),
    ("gurugram", "Gurugram", "Haryana"), ("gurgaon", "Gurugram", "Haryana"), ("faridabad", "Faridabad", "Haryana"),
    ("noida", "Noida", "Uttar Pradesh"), ("greater noida", "Noida", "Uttar Pradesh"), ("ghaziabad", "Ghaziabad", "Uttar Pradesh"),
    ("lucknow", "Lucknow", "Uttar Pradesh"), ("kanpur", "Kanpur", "Uttar Pradesh"),
    ("kolkata", "Kolkata", "West Bengal"), ("calcutta", "Kolkata", "West Bengal"),
    ("ahmedabad", "Ahmedabad", "Gujarat"), ("gandhinagar", "Gandhinagar", "Gujarat"), ("vadodara", "Vadodara", "Gujarat"), ("surat", "Surat", "Gujarat"),
    ("jaipur", "Jaipur", "Rajasthan"), ("chandigarh", "Chandigarh", "Chandigarh"), ("mohali", "Mohali", "Punjab"),
    ("sahibzada ajit singh nagar", "Mohali", "Punjab"), ("ludhiana", "Ludhiana", "Punjab"),
    ("indore", "Indore", "Madhya Pradesh"), ("bhopal", "Bhopal", "Madhya Pradesh"),
    ("thiruvananthapuram", "Thiruvananthapuram", "Kerala"), ("trivandrum", "Thiruvananthapuram", "Kerala"), ("kochi", "Kochi", "Kerala"), ("cochin", "Kochi", "Kerala"),
    ("visakhapatnam", "Visakhapatnam", "Andhra Pradesh"), ("vizag", "Visakhapatnam", "Andhra Pradesh"), ("vijayawada", "Vijayawada", "Andhra Pradesh"),
    ("bhubaneswar", "Bhubaneswar", "Odisha"), ("patna", "Patna", "Bihar"), ("ranchi", "Ranchi", "Jharkhand"), ("guwahati", "Guwahati", "Assam"),
    ("dehradun", "Dehradun", "Uttarakhand"), ("raipur", "Raipur", "Chhattisgarh"), ("goa", "Goa", "Goa"),
    # States, for locations like "Maharashtra, India" or "Karnataka"
    ("karnataka", None, "Karnataka"), ("telangana", None, "Telangana"), ("tamil nadu", None, "Tamil Nadu"), ("maharashtra", None, "Maharashtra"),
    ("haryana", None, "Haryana"), ("uttar pradesh", None, "Uttar Pradesh"), ("west bengal", None, "West Bengal"), ("gujarat", None, "Gujarat"),
    ("rajasthan", None, "Rajasthan"), ("punjab", None, "Punjab"), ("kerala", None, "Kerala"), ("andhra pradesh", None, "Andhra Pradesh"),
    ("madhya pradesh", None, "Madhya Pradesh"), ("odisha", None, "Odisha"),
]

(
    spark.createDataFrame(INDIA_LOCATIONS, "alias STRING, city STRING, state STRING")
    .selectExpr("*", f"'{REF_VERSION}' AS ref_version")
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.ops.ref_india_locations")
)
print(f"ref_india_locations: {len(INDIA_LOCATIONS)} rows")

## Verify

In [ ]:
display(spark.sql(f"""
SELECT table_schema, table_name, table_type
FROM {CATALOG}.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold', 'ops')
ORDER BY table_schema, table_name
"""))